# Lumen AI — Optimizer Evaluation & Executive Plots

Databricks-friendly notebook for customer-level BAU vs optimizer comparison.

This notebook assumes one row per customer with:
- BAU historical channel assignment columns
- model-predicted lift/score columns per channel
- optimizer decision columns for scenario 1 and scenario 2
- historical conversion label
- PRISM any-credit-card propensity score

Main outputs:
1. Channel allocation by PRISM propensity decile for BAU, scenario 1, scenario 2
2. Match-group decile validation where BAU agrees with optimizer
3. Non-match model-based expected uplift versus BAU
4. Summary tables saved to Excel/CSV


In [ ]:
# Databricks setup
# If running on a cluster where these are already installed, no action is needed.

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional: improve notebook display in Databricks
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1. Configuration

Edit this cell first. The only assumptions are the channel suffixes and column prefixes.

Example expected columns for channel `leads`:
- `bau_leads`
- `lift_leads`
- `opt1_leads`
- `opt2_leads`


In [ ]:
CHANNELS = [
    "leads",
    "email",
    "direct_mail",
    "phone",
    "mobile",
    "branch",
    "web",
]

# Replace these with your real table / output paths.
INPUT_TABLE = "your_catalog.your_schema.customer_optimizer_outputs"  # Spark table name
OUTPUT_DIR = "/dbfs/FileStore/lumen_ai_plot_outputs"                # DBFS-backed local path

# Column configuration
BAU_PREFIX = "bau_"          # e.g., bau_leads, bau_email, ...
LIFT_PREFIX = "lift_"        # e.g., lift_leads, lift_email, ...

SCENARIO_PREFIXES = {
    "scenario_1_unconstrained": "opt1_",      # e.g., opt1_leads, opt1_email, ...
    "scenario_2_leads_cap_30k": "opt2_",      # e.g., opt2_leads, opt2_email, ...
}

CONVERSION_COL = "converted"
PRISM_PROPENSITY_COL = "prism_any_cc_propensity"
CUSTOMER_ID_COL = "customer_id"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR)


## 2. Helper functions

These functions are intentionally kept inside the notebook so you can upload one file to Databricks and run it directly.


In [ ]:
"""
Reusable plotting/evaluation utilities for Lumen AI next-best-channel optimizer outputs.

Expected customer-level table (Pandas DataFrame):
- one row per customer
- historical BAU channel assignment as one-hot binary columns
- model-predicted lift/score per channel as float columns
- optimizer scenario decisions as one-hot binary columns
- conversion label column
- PRISM any-credit-card propensity column

The utilities assume one channel per policy/customer after tie-breaking. If your BAU has
multiple active channels, set multi_channel_strategy='max_lift' or 'first' in infer_policy_channel().
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


@dataclass(frozen=True)
class LumenPlotConfig:
    channels: Sequence[str]
    bau_prefix: str = "bau_"
    lift_prefix: str = "lift_"
    scenario_prefixes: Dict[str, str] = None
    conversion_col: str = "converted"
    prism_propensity_col: str = "prism_any_cc_propensity"
    customer_id_col: Optional[str] = "customer_id"
    output_dir: str = "lumen_plots"

    def __post_init__(self):
        if self.scenario_prefixes is None:
            object.__setattr__(
                self,
                "scenario_prefixes",
                {
                    "scenario_1_unconstrained": "s1_",
                    "scenario_2_constrained": "s2_",
                },
            )


def _cols(prefix: str, channels: Sequence[str]) -> List[str]:
    return [f"{prefix}{ch}" for ch in channels]


def validate_input(df: pd.DataFrame, cfg: LumenPlotConfig) -> None:
    required = [cfg.conversion_col, cfg.prism_propensity_col]
    required += _cols(cfg.bau_prefix, cfg.channels)
    required += _cols(cfg.lift_prefix, cfg.channels)
    for prefix in cfg.scenario_prefixes.values():
        required += _cols(prefix, cfg.channels)
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def infer_policy_channel(
    df: pd.DataFrame,
    prefix: str,
    channels: Sequence[str],
    lift_prefix: Optional[str] = None,
    multi_channel_strategy: str = "max_lift",
    no_channel_label: str = "none",
) -> pd.Series:
    """
    Convert one-hot policy columns into one chosen channel per customer.

    multi_channel_strategy:
    - 'max_lift': if multiple policy columns are 1, choose the active channel with highest model lift.
    - 'first': choose the first active channel in cfg.channels order.
    - 'error': raise if more than one active channel exists.
    """
    policy_cols = _cols(prefix, channels)
    active = df[policy_cols].fillna(0).astype(float).values > 0
    n_active = active.sum(axis=1)

    if multi_channel_strategy == "error" and np.any(n_active > 1):
        raise ValueError("Some rows have multiple active channels for policy prefix " + prefix)

    chosen_idx = np.full(len(df), -1, dtype=int)

    if multi_channel_strategy == "first":
        has_any = n_active > 0
        chosen_idx[has_any] = active[has_any].argmax(axis=1)
    elif multi_channel_strategy == "max_lift":
        if lift_prefix is None:
            raise ValueError("lift_prefix is required when multi_channel_strategy='max_lift'")
        lift_cols = _cols(lift_prefix, channels)
        lift_values = df[lift_cols].astype(float).values.copy()
        lift_values[~active] = -np.inf
        has_any = n_active > 0
        chosen_idx[has_any] = np.nanargmax(lift_values[has_any], axis=1)
    elif multi_channel_strategy != "error":
        raise ValueError("multi_channel_strategy must be one of: max_lift, first, error")

    labels = np.array(list(channels), dtype=object)
    out = np.full(len(df), no_channel_label, dtype=object)
    valid = chosen_idx >= 0
    out[valid] = labels[chosen_idx[valid]]
    return pd.Series(out, index=df.index, name=f"{prefix}chosen_channel")


def add_deciles(
    df: pd.DataFrame,
    score_col: str,
    decile_col: str,
    highest_is_1: bool = True,
) -> pd.DataFrame:
    out = df.copy()
    rank_score = -out[score_col] if highest_is_1 else out[score_col]
    out[decile_col] = pd.qcut(rank_score.rank(method="first"), 10, labels=range(1, 11)).astype(int)
    return out


def prepare_policy_columns(
    df: pd.DataFrame,
    cfg: LumenPlotConfig,
    multi_channel_strategy: str = "max_lift",
) -> pd.DataFrame:
    """Adds bau_channel and one chosen channel column per optimizer scenario."""
    validate_input(df, cfg)
    out = df.copy()
    out["bau_channel"] = infer_policy_channel(
        out, cfg.bau_prefix, cfg.channels, cfg.lift_prefix, multi_channel_strategy
    )
    for scenario_name, prefix in cfg.scenario_prefixes.items():
        out[f"{scenario_name}_channel"] = infer_policy_channel(
            out, prefix, cfg.channels, cfg.lift_prefix, multi_channel_strategy
        )
    return out


def chosen_lift(
    df: pd.DataFrame,
    chosen_channel_col: str,
    cfg: LumenPlotConfig,
    output_col: str,
) -> pd.Series:
    """Return model lift associated with the chosen channel for each row."""
    lift_map = {ch: f"{cfg.lift_prefix}{ch}" for ch in cfg.channels}
    values = np.full(len(df), np.nan)
    for ch, col in lift_map.items():
        mask = df[chosen_channel_col].eq(ch).values
        values[mask] = df.loc[mask, col].astype(float).values
    return pd.Series(values, index=df.index, name=output_col)


def make_match_table(
    df: pd.DataFrame,
    cfg: LumenPlotConfig,
    scenario_name: str,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Returns:
    - row-level df with match flag and chosen lifts
    - decile summary for BAU/model match group
    - aggregate summary for match and non-match groups
    """
    scenario_col = f"{scenario_name}_channel"
    if scenario_col not in df.columns:
        raise ValueError(f"Run prepare_policy_columns first; missing {scenario_col}")

    out = df.copy()
    out[f"{scenario_name}_matches_bau"] = out[scenario_col].eq(out["bau_channel"])
    out["bau_lift"] = chosen_lift(out, "bau_channel", cfg, "bau_lift")
    out[f"{scenario_name}_lift"] = chosen_lift(out, scenario_col, cfg, f"{scenario_name}_lift")
    out[f"{scenario_name}_incremental_lift_vs_bau"] = out[f"{scenario_name}_lift"] - out["bau_lift"]

    match_df = out[out[f"{scenario_name}_matches_bau"]].copy()
    if len(match_df) > 0:
        match_df = add_deciles(
            match_df,
            score_col=f"{scenario_name}_lift",
            decile_col=f"{scenario_name}_lift_decile",
            highest_is_1=True,
        )
        decile_summary = (
            match_df.groupby(f"{scenario_name}_lift_decile", observed=True)
            .agg(
                customers=(cfg.conversion_col, "size"),
                converters=(cfg.conversion_col, "sum"),
                conversion_rate=(cfg.conversion_col, "mean"),
                avg_model_lift=(f"{scenario_name}_lift", "mean"),
            )
            .reset_index()
        )
    else:
        decile_summary = pd.DataFrame()

    agg_summary = (
        out.groupby(f"{scenario_name}_matches_bau", observed=True)
        .agg(
            customers=(cfg.conversion_col, "size"),
            converters=(cfg.conversion_col, "sum"),
            actual_conversion_rate=(cfg.conversion_col, "mean"),
            avg_bau_lift=("bau_lift", "mean"),
            avg_model_lift=(f"{scenario_name}_lift", "mean"),
            avg_incremental_lift=(f"{scenario_name}_incremental_lift_vs_bau", "mean"),
            total_expected_incremental_conversions=(f"{scenario_name}_incremental_lift_vs_bau", "sum"),
        )
        .reset_index()
        .rename(columns={f"{scenario_name}_matches_bau": "matches_bau"})
    )
    agg_summary["expected_lift_pct_vs_bau"] = np.where(
        agg_summary["avg_bau_lift"].abs() > 1e-12,
        agg_summary["avg_incremental_lift"] / agg_summary["avg_bau_lift"] * 100,
        np.nan,
    )
    return out, decile_summary, agg_summary


def channel_distribution_by_decile(
    df: pd.DataFrame,
    cfg: LumenPlotConfig,
    policy_channel_col: str,
    decile_col: str = "prism_decile",
) -> pd.DataFrame:
    """Percentage channel mix by PRISM propensity decile for one policy."""
    if decile_col not in df.columns:
        df = add_deciles(df, cfg.prism_propensity_col, decile_col, highest_is_1=True)
    counts = pd.crosstab(df[decile_col], df[policy_channel_col], normalize="index") * 100
    return counts.reindex(index=range(1, 11), columns=list(cfg.channels) + ["none"], fill_value=0)


def plot_channel_distribution_by_decile(
    dist: pd.DataFrame,
    title: str,
    output_path: Optional[str] = None,
    figsize: Tuple[int, int] = (11, 6),
) -> plt.Figure:
    """Stacked bar chart: channel percentage by PRISM decile."""
    fig, ax = plt.subplots(figsize=figsize)
    dist.plot(kind="bar", stacked=True, ax=ax, width=0.85)
    ax.set_title(title)
    ax.set_xlabel("PRISM propensity decile (1 = highest)")
    ax.set_ylabel("Customers allocated to channel (%)")
    ax.legend(title="Channel", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.set_ylim(0, 100)
    fig.tight_layout()
    if output_path:
        fig.savefig(output_path, dpi=200, bbox_inches="tight")
    return fig


def plot_match_decile_validation(
    decile_summary: pd.DataFrame,
    scenario_name: str,
    output_path: Optional[str] = None,
    figsize: Tuple[int, int] = (9, 5),
) -> plt.Figure:
    """For BAU/model matches: actual conversion rate by model-lift decile."""
    if decile_summary.empty:
        raise ValueError("decile_summary is empty; no matched rows to plot.")
    dec_col = f"{scenario_name}_lift_decile"
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(decile_summary[dec_col], decile_summary["conversion_rate"] * 100, marker="o")
    ax.set_title(f"Actual conversion concentration where BAU matches {scenario_name}")
    ax.set_xlabel("Model lift decile among matched customers (1 = highest)")
    ax.set_ylabel("Actual conversion rate (%)")
    ax.set_xticks(range(1, 11))
    ax.grid(True, axis="y", alpha=0.3)
    for _, row in decile_summary.iterrows():
        ax.annotate(
            f"n={int(row['customers']):,}",
            (row[dec_col], row["conversion_rate"] * 100),
            textcoords="offset points",
            xytext=(0, 6),
            ha="center",
            fontsize=8,
        )
    fig.tight_layout()
    if output_path:
        fig.savefig(output_path, dpi=200, bbox_inches="tight")
    return fig


def plot_nonmatch_expected_uplift(
    row_df: pd.DataFrame,
    cfg: LumenPlotConfig,
    scenario_name: str,
    output_path: Optional[str] = None,
    figsize: Tuple[int, int] = (8, 5),
) -> Tuple[plt.Figure, pd.DataFrame]:
    """
    For non-match rows, compare predicted BAU lift vs optimizer-chosen lift.
    This is model-based counterfactual evidence, not observed causal proof.
    """
    match_col = f"{scenario_name}_matches_bau"
    scen_lift = f"{scenario_name}_lift"
    inc_col = f"{scenario_name}_incremental_lift_vs_bau"
    non = row_df[~row_df[match_col]].copy()
    if non.empty:
        raise ValueError("No non-match rows to plot.")

    summary = pd.DataFrame(
        {
            "metric": ["BAU selected channel", "Optimizer selected channel"],
            "avg_predicted_lift": [non["bau_lift"].mean(), non[scen_lift].mean()],
            "total_expected_conversions": [non["bau_lift"].sum(), non[scen_lift].sum()],
            "customers": [len(non), len(non)],
        }
    )
    expected_incremental = non[inc_col].sum()
    expected_pct = expected_incremental / max(abs(non["bau_lift"].sum()), 1e-12) * 100
    summary.attrs["expected_incremental_conversions"] = expected_incremental
    summary.attrs["expected_pct_vs_bau"] = expected_pct

    fig, ax = plt.subplots(figsize=figsize)
    ax.bar(summary["metric"], summary["avg_predicted_lift"] * 100)
    ax.set_title(f"Model-based expected lift in non-match cases: {scenario_name}")
    ax.set_ylabel("Average predicted lift / conversion probability (%)")
    ax.text(
        0.5,
        0.95,
        f"Expected incremental conversions: {expected_incremental:,.0f}\nExpected % improvement vs BAU: {expected_pct:,.1f}%",
        transform=ax.transAxes,
        ha="center",
        va="top",
        bbox={"boxstyle": "round", "alpha": 0.15},
    )
    fig.tight_layout()
    if output_path:
        fig.savefig(output_path, dpi=200, bbox_inches="tight")
    return fig, summary


def run_all_lumen_plots(
    df: pd.DataFrame,
    cfg: LumenPlotConfig,
    multi_channel_strategy: str = "max_lift",
) -> Dict[str, pd.DataFrame]:
    """End-to-end driver: prepares columns, creates plots, and returns summary tables."""
    out_dir = Path(cfg.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    prepared = prepare_policy_columns(df, cfg, multi_channel_strategy=multi_channel_strategy)
    prepared = add_deciles(prepared, cfg.prism_propensity_col, "prism_decile", highest_is_1=True)

    tables: Dict[str, pd.DataFrame] = {"prepared_customer_level": prepared}

    # Channel mix by PRISM decile: BAU + each scenario as separate plots to avoid clutter.
    policy_cols = {"bau": "bau_channel"}
    policy_cols.update({name: f"{name}_channel" for name in cfg.scenario_prefixes})
    for policy_name, col in policy_cols.items():
        dist = channel_distribution_by_decile(prepared, cfg, col, decile_col="prism_decile")
        tables[f"channel_mix_{policy_name}"] = dist.reset_index().rename(columns={"index": "prism_decile"})
        plot_channel_distribution_by_decile(
            dist,
            title=f"Channel allocation by PRISM decile — {policy_name}",
            output_path=str(out_dir / f"channel_mix_by_prism_decile__{policy_name}.png"),
        )
        plt.close()

    for scenario_name in cfg.scenario_prefixes:
        row_df, decile_summary, agg_summary = make_match_table(prepared, cfg, scenario_name)
        tables[f"match_deciles_{scenario_name}"] = decile_summary
        tables[f"match_summary_{scenario_name}"] = agg_summary
        plot_match_decile_validation(
            decile_summary,
            scenario_name,
            output_path=str(out_dir / f"match_decile_validation__{scenario_name}.png"),
        )
        plt.close()
        _, nonmatch_summary = plot_nonmatch_expected_uplift(
            row_df,
            cfg,
            scenario_name,
            output_path=str(out_dir / f"nonmatch_expected_uplift__{scenario_name}.png"),
        )
        plt.close()
        tables[f"nonmatch_expected_uplift_{scenario_name}"] = nonmatch_summary

    return tables


## 3. Load data

Use the Spark table path if your data is already in Databricks. For very large data, filter/sample first or aggregate in Spark before converting to Pandas. The plotting functions expect a Pandas DataFrame.


In [ ]:
# Option A: read from a Spark table
# df = spark.table(INPUT_TABLE).toPandas()

# Option B: if you already built a Spark DataFrame upstream, uncomment and replace `spark_df`
# df = spark_df.toPandas()

# Option C: read from a file in DBFS/local path
# df = pd.read_parquet("/dbfs/path/to/customer_optimizer_outputs.parquet")

# Safety placeholder: fail clearly until one of the options above is enabled.
try:
    df
except NameError:
    raise NameError("Create `df` first by uncommenting one of the loading options above.")

print(df.shape)
display(df.head())


## 4. Run all plots and summary tables


In [ ]:
cfg = LumenPlotConfig(
    channels=CHANNELS,
    bau_prefix=BAU_PREFIX,
    lift_prefix=LIFT_PREFIX,
    scenario_prefixes=SCENARIO_PREFIXES,
    conversion_col=CONVERSION_COL,
    prism_propensity_col=PRISM_PROPENSITY_COL,
    customer_id_col=CUSTOMER_ID_COL,
    output_dir=OUTPUT_DIR,
)

tables = run_all_lumen_plots(
    df=df,
    cfg=cfg,
    multi_channel_strategy="max_lift",  # use "first" if BAU columns are multi-touch and you want first active channel
)

print("Generated tables:")
for name, table in tables.items():
    print(f"- {name}: {table.shape}")


## 5. Display key tables


In [ ]:
for name, table in tables.items():
    if name != "prepared_customer_level":
        print("
" + "=" * 100)
        print(name)
        display(table)


## 6. Save Excel and CSV outputs


In [ ]:
excel_path = f"{OUTPUT_DIR}/lumen_ai_plot_summary_tables.xlsx"

with pd.ExcelWriter(excel_path) as writer:
    for name, table in tables.items():
        if name == "prepared_customer_level":
            continue
        table.to_excel(writer, sheet_name=name[:31], index=False)

# Also save individual CSVs, easier to inspect in Databricks/FileStore.
for name, table in tables.items():
    if name == "prepared_customer_level":
        continue
    table.to_csv(f"{OUTPUT_DIR}/{name}.csv", index=False)

print("Saved Excel:", excel_path)
print("Saved CSVs and PNG plots under:", OUTPUT_DIR)


## 7. Optional: inspect customer-level match/non-match rows

This is useful for sanity checks and executive examples. You can filter by scenario and inspect customers where BAU and optimizer disagree.


In [ ]:
prepared = tables["prepared_customer_level"]

scenario_name = "scenario_1_unconstrained"
row_df, decile_summary, agg_summary = make_match_table(prepared, cfg, scenario_name)

nonmatch = row_df[~row_df[f"{scenario_name}_matches_bau"]].copy()
nonmatch = nonmatch.sort_values(f"{scenario_name}_incremental_lift_vs_bau", ascending=False)

cols_to_show = [
    CUSTOMER_ID_COL,
    "bau_channel",
    f"{scenario_name}_channel",
    "bau_lift",
    f"{scenario_name}_lift",
    f"{scenario_name}_incremental_lift_vs_bau",
    CONVERSION_COL,
    PRISM_PROPENSITY_COL,
]
cols_to_show = [c for c in cols_to_show if c in nonmatch.columns]

display(nonmatch[cols_to_show].head(50))
